# 01.1 — Parquet → BigQuery (carga lazy OOM-safe)

Lee el Parquet particionado (Freddie Mac SFLLD) desde Drive y lo carga a **BigQuery** en modo lazy:

```
parquet/year=YYYY/*.parquet  →  BigQuery staging table
```

**Estrategia anti-OOM:**
- Lectura **partición por partición** (year=1999, year=2000, …)
- Cada partición se lee, convierte a pandas y se carga a BigQuery
- Liberación explícita de memoria tras cada batch
- Sin RAPIDS: PyArrow + pandas para minimizar dependencias

## 1. Setup Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado.')

In [ ]:
!pip install -q pyarrow pandas db-dtypes
print('Dependencias OK.')

## 2. Autenticación BigQuery

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Autenticado.')

In [ ]:
from google.cloud import bigquery

PROJECT_ID = 'tareas-303400'   # ← Cambiar por tu proyecto GCP
DATASET_ID = 'mortgage_risk'   # ← Dataset en BigQuery (crear si no existe)
TABLE_ID   = 'stg_loans'       # ← Tabla staging

client = bigquery.Client(project=PROJECT_ID)
full_table_id = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}'

# Crear dataset si no existe
dataset_ref = bigquery.Dataset(f'{PROJECT_ID}.{DATASET_ID}')
try:
    client.get_dataset(dataset_ref)
except Exception:
    client.create_dataset(dataset_ref)
    print(f'Dataset {DATASET_ID} creado.')

print(f'Cliente BigQuery: {PROJECT_ID}')
print(f'Tabla destino: {full_table_id}')

## 3. Configuración rutas Parquet

In [ ]:
import os
import gc
from pathlib import Path

DRIVE_BASE   = '/content/drive/MyDrive'
PARQUET_BASE = f'{DRIVE_BASE}/mortgage-risk/parquet'

if not os.path.isdir(PARQUET_BASE):
    raise FileNotFoundError(f'No existe: {PARQUET_BASE}')

year_dirs = sorted([d for d in os.listdir(PARQUET_BASE) if d.startswith('year=')])
print(f'Particiones (year=): {len(year_dirs)}')
print(f'Ejemplos: {year_dirs[:5]} ... {year_dirs[-3:] if len(year_dirs)>5 else ""}')

## 4. Carga lazy partición por partición

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import time

BATCH_SIZE_FILES = 20   # Máx. archivos parquet por batch (ajustar si OOM)
WRITE_TRUNCATE_FIRST = True

total_rows = 0
t0 = time.time()

for idx, year_dir in enumerate(year_dirs):
    year_path = os.path.join(PARQUET_BASE, year_dir)
    files = sorted(Path(year_path).glob('*.parquet'))
    if not files:
        continue

    # Leer en sub-batches si hay muchos archivos
    for start in range(0, len(files), BATCH_SIZE_FILES):
        batch_files = files[start:start + BATCH_SIZE_FILES]
        table = pq.read_table(batch_files)
        df = table.to_pandas()
        del table; gc.collect()

        if len(df) == 0:
            continue

        write_mode = bigquery.WriteDisposition.WRITE_TRUNCATE if (idx == 0 and start == 0 and WRITE_TRUNCATE_FIRST) else bigquery.WriteDisposition.WRITE_APPEND
        job_config = bigquery.LoadJobConfig(
            write_disposition=write_mode,
            autodetect=True,
        )

        load_job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
        load_job.result()
        total_rows += len(df)
        del df; gc.collect()

    print(f'  {year_dir}: {len(files)} archivos → {total_rows:,} filas acumuladas')

elapsed = time.time() - t0
print(f'\n✓ Carga completada: {total_rows:,} filas en {elapsed:.1f}s')

## 5. Verificación

In [ ]:
destination_table = client.get_table(full_table_id)
print(f'Tabla: {full_table_id}')
print(f'Filas cargadas: {destination_table.num_rows:,}')
print(f'Columnas: {len(destination_table.schema)}')